# Treasury Yield Data Pipeline

This notebook builds a monthly panel dataset (Jan 2015 – Dec 2024) of US
macroeconomic indicators used to model US 10-Year Treasury yields.

**Data sources:**
- US 10Y, Federal Funds Rate, CPI, Unemployment Rate: FRED (CSV exports)
- S&P 500, Gold, USD Index: Yahoo Finance via yfinance
- China 10Y: separate CSV export

**Output:** `Merged_Data.csv` with 120 monthly observations across 9 variables,
used as input to econometric analysis in R (see `analysis.Rmd`).

# Import Data and Basic Timeframe Alignment

## Market Index (S&P 500)

In [ ]:
import yfinance as yf
import pandas as pd

In [ ]:
sp500 = yf.download("^GSPC", start="2015-01-01", end="2024-12-31", interval="1d")
sp500.index = pd.to_datetime(sp500.index)

In [ ]:
print(sp500.columns)

In [ ]:
sp500_eom = sp500.resample('M').last() #Select only the last business day of each month
sp500_eom= sp500_eom[['Close']].rename(columns={'Close': 'SP_500'})
print(sp500_eom.head)

In [ ]:
sp500_eom.to_csv("sp500_monthly.csv")

## Precious Metal Price Index (Gold Price)

In [ ]:
gold = yf.download("GC=F", start="2015-01-01", end="2024-12-31", interval="1d")
gold.index = pd.to_datetime(gold.index)

In [ ]:
gold_eom = gold.resample('M').last() #Select only the last business day of each month
gold_eom = gold_eom[['Close']].rename(columns={'Close': 'Gold_Price'})
print(gold_eom.head)

In [ ]:
gold_eom.to_csv("gold_monthly.csv")
print(gold_eom.head)

## US Currency Index (USD Value)

In [ ]:
usd = yf.download("DX-Y.NYB", start="2015-01-01", end="2024-12-31", interval="1d")
usd.index = pd.to_datetime(usd.index)

In [ ]:
usd_eom = usd.resample('M').last() #Select only the last business day of each month
usd_eom = usd_eom[['Close']].rename(columns={'Close': 'USD_Index'}) #Select only the last business day of each month

In [ ]:
usd_eom.to_csv("usd_monthly.csv")
print(usd_eom.head)

## Interest Rate (Federal Funds Rate)

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
file_path = "/content/drive/My Drive/Colab_Data/FEDFUNDS.csv"

In [ ]:
ffr = pd.read_csv(file_path, parse_dates=["observation_date"])
ffr.set_index("observation_date", inplace=True)

In [ ]:
ffr.index = ffr.index - pd.DateOffset(days=1)

In [ ]:
ffr.rename(columns={"FEDFUNDS": "Fed_Funds_Rate"}, inplace=True)

In [ ]:
print(ffr.head)

In [ ]:
output_path = "/content/drive/My Drive/Colab_Data/FEDFUNDS_Adjusted.csv"
ffr.to_csv(output_path)
ffr.to_csv("fed_funds_rate.csv")

## Inflation Rate (based on Consumer Price Index)

In [ ]:
file_path = "/content/drive/My Drive/Colab_Data/CPIAUCSL.csv"

In [ ]:
cpi = pd.read_csv(file_path, parse_dates=["observation_date"])
cpi.set_index("observation_date", inplace=True)

In [ ]:
cpi.index = cpi.index - pd.DateOffset(days=1)

In [ ]:
cpi.rename(columns={"CPIAUCSL": "CPI"}, inplace=True)

In [ ]:
print(cpi.head)

In [ ]:
cpi["Inflation_Rate"] = cpi["CPI"].pct_change() * 100

In [ ]:
# pct_change() produces NaN for the first observation (Jan 2015) since
# we don't have December 2014 CPI loaded. Manually fill with the value
# computed from FRED CPI data outside this notebook.
# Note: this affects 1 observation; the final model also drops early
# rows due to 2-month inflation lag, minimizing impact.
cpi.iloc[0, cpi.columns.get_loc("Inflation_Rate")] = 0.2535

In [ ]:
print(cpi.head)

In [ ]:
output_path = "/content/drive/My Drive/Colab_Data/CPIAUCSL_Adjusted.csv"
cpi.to_csv(output_path)
cpi.to_csv("cpi.csv")

## Foreign Bond Yields (China 10Y)

In [ ]:
file_path = "/content/drive/My Drive/Colab_Data/china10y.csv"

In [ ]:
cn10y = pd.read_csv(file_path, parse_dates=["Date"])
cn10y.set_index("Date", inplace=True)

In [ ]:
cn10y_eom = cn10y.resample('M').last() #Select only the last business day of each month
cn10y_eom = cn10y_eom[['Close']].rename(columns={'Close': 'China_10Y'})
print(cn10y_eom.head)

In [ ]:
cn10y_eom.to_csv("cn10y_monthly.csv")
print(cn10y_eom.head)

## Unemployment Rate

In [ ]:
file_path = "/content/drive/My Drive/Colab_Data/UNRATE.csv"

In [ ]:
unrate = pd.read_csv(file_path, parse_dates=["observation_date"])
unrate.set_index("observation_date", inplace=True)

In [ ]:
unrate.index = unrate.index - pd.DateOffset(days=1)

In [ ]:
print(unrate.head)

In [ ]:
output_path = "/content/drive/My Drive/Colab_Data/UNRATE_Adjusted.csv"
unrate.to_csv(output_path)
unrate.to_csv("unrate.csv")

## Response variable: US10Y

In [ ]:
file_path = "/content/drive/My Drive/Colab_Data/DGS10.csv"

In [ ]:
us10 = pd.read_csv(file_path, parse_dates=["observation_date"])
us10.set_index("observation_date", inplace=True)

In [ ]:
us10_eom = us10.resample('M').last() #Select only the last business day of each month
us10_eom= us10_eom[['DGS10']].rename(columns={'DGS10': 'US_10Y'})
print(us10_eom.head)

In [ ]:
us10_eom = us10_eom.drop(us10_eom.index[-1])
print(us10_eom.head)

In [ ]:
output_path = "/content/drive/My Drive/Colab_Data/DGS10_Adjusted.csv"
us10_eom.to_csv(output_path)
us10_eom.to_csv("us10_monthly.csv")

## Additional Action: Save CSVs to Google Drive

In [ ]:
import shutil

In [ ]:
files = ["cn10y_monthly.csv", "gold_monthly.csv", "sp500_monthly.csv", "usd_monthly.csv"]
# I accidentally deleted the local files, so this step is to make a copy from GDrive back to here

In [ ]:
source_folder = "/content/drive/My Drive/Colab_Data/"

In [ ]:
for file in files:
    shutil.copy(source_folder + file, file)

# Clean a Few Datasets

# Merge Datasets

In [ ]:
cn10y = pd.read_csv("cn10y_monthly.csv", parse_dates=["Date"], index_col="Date")
cpi = pd.read_csv("cpi.csv", parse_dates=["observation_date"], index_col="observation_date")
fed_funds_rate = pd.read_csv("fed_funds_rate.csv", parse_dates=["observation_date"], index_col="observation_date")
gold = pd.read_csv("gold_monthly.csv", skiprows=2, parse_dates=["Date"], index_col="Date")
sp500 = pd.read_csv("sp500_monthly.csv", skiprows=2, parse_dates=["Date"], index_col="Date")
unrate = pd.read_csv("unrate.csv", parse_dates=["observation_date"], index_col="observation_date")
us10y = pd.read_csv("us10_monthly.csv", parse_dates=["observation_date"], index_col="observation_date")  # Response variable
usd = pd.read_csv("usd_monthly.csv",skiprows=2, parse_dates=["Date"], index_col="Date")

In [ ]:
merged_data = cn10y.merge(cpi, on="Date", how="inner")\
                   .merge(fed_funds_rate, on="Date", how="inner")\
                   .merge(gold, on="Date", how="inner")\
                   .merge(sp500, on="Date", how="inner")\
                   .merge(unrate, on="Date", how="inner")\
                   .merge(us10y, on="Date", how="inner")\
                   .merge(usd, on="Date", how="inner")